In [ ]:
import arviz as az
import seaborn as sns
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
import itertools
from IPython.display import display, Markdown
import pandas as pd
from matplotlib import pyplot as plt
import pycountry

from emu_renewal.inputs import OUTPUTS_PATH, get_google_mobility, get_apple_mobility
from emu_renewal.utils import get_countries_by_continent
from emu_renewal.plotting import get_standard_subplot

In [ ]:
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
countries = ls(job_path)
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
mob_type = "g_mob"

In [ ]:
def get_idatas_for_mob_type(job_path, countries, mob_type):
    country_idatas = {}
    for iso3 in countries:
        country = pycountry.countries.lookup(iso3).name
        try:
            country_idatas[iso3] = az.from_netcdf(job_path / iso3 / mob_type / "idata_filtered.nc")
        except:
            print(f"{mob_type} unavailable for {country}")
    return country_idatas

In [ ]:
for cont in countries_by_cont:
    idatas = get_idatas_for_mob_type(job_path, countries_by_cont[cont], "g_mob")
    fig, axes = get_standard_subplot(len(idatas), 4)
    axes = axes.ravel()
    for c, iso3 in enumerate(idatas):
        idata = idatas[iso3]
        country = pycountry.countries.lookup(iso3).name
        ax = axes[c]
        az.plot_posterior(idata.posterior["mob_exp"], ax=ax)
        ax.set_title(country)
    fig.tight_layout()